# Refund Report Generator
This notebook defines a reusable function that generates `refund_report.csv` from the CSV source tables.

In [ ]:
import os
import pandas as pd

In [ ]:
def generate_refund_report(data_dir='data', output_dir='output'):
    os.makedirs(output_dir, exist_ok=True)

    order_item_refunds = pd.read_csv(f'{data_dir}/order_item_refunds.csv')
    order_items = pd.read_csv(f'{data_dir}/order_items.csv')
    orders = pd.read_csv(f'{data_dir}/orders.csv')
    products = pd.read_csv(f'{data_dir}/products.csv')
    website_pageviews = pd.read_csv(f'{data_dir}/website_pageviews.csv')

    refunds_with_items = order_item_refunds.merge(
        order_items[['order_item_id', 'product_id']],
        on='order_item_id',
        how='left'
    )

    refunds_with_items = refunds_with_items.rename(columns={'created_at': 'refund_created_at'})

    refunds_with_products = refunds_with_items.merge(
        products[['product_id', 'product_name']],
        on='product_id',
        how='left'
    )

    refunds_with_orders = refunds_with_products.merge(
        orders[['order_id', 'created_at', 'website_session_id']],
        on='order_id',
        how='left'
    )
    refunds_with_orders = refunds_with_orders.rename(columns={'created_at': 'order_created_at'})

    pageviews_first = website_pageviews.groupby('website_session_id')['pageview_url'].first().reset_index()

    refunds_complete = refunds_with_orders.merge(
        pageviews_first,
        on='website_session_id',
        how='left'
    )

    refund_report = pd.DataFrame({
        'refund_created_date': pd.to_datetime(refunds_complete['refund_created_at']).dt.date,
        'order_id': refunds_complete['order_id'],
        'refund_amount': refunds_complete['refund_amount_usd'],
        'product_id': refunds_complete['product_id'],
        'product_name': refunds_complete['product_name'],
        'order_created_date': pd.to_datetime(refunds_complete['order_created_at']).dt.date,
        'pageview_url': refunds_complete['pageview_url'],
    })

    refund_created = pd.to_datetime(refunds_complete['refund_created_at'])
    order_created = pd.to_datetime(refunds_complete['order_created_at'])
    refund_report['order_duration'] = (
        (refund_created - order_created).dt.total_seconds() / 60
    ).astype('Int64')

    refund_report.to_csv(f'{output_dir}/refund_report.csv', index=False)
    return refund_report

In [ ]:
report = generate_refund_report()
print(f'Written output/refund_report.csv with {len(report)} rows.')
report.head()